In [ ]:
#Naive Bayes: Prepared Remarks vs. Q&A

##Enter a new sentence - it should be classified as either coming from prepared remarks, or Q&A

#Use only responses to questions in your Q&A model

#Build the naive bayes model

#Show the naive bayes scores, etc…

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np
import math
import re

BasePath = '/content/drive/MyDrive/Text_Mining/earnings_calls/'

DF1 = pd.read_csv(BasePath + 'DF1.csv')
DF2 = pd.read_csv(BasePath + 'DF2.csv')

def words(text):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower().replace('’', "'")
    )

data = DF2.dropna(
    subset=['EarningsCallID', 'SpeakerName',
            'PreparedOrQA', 'Text']
).copy()

data['SpeakerName'] = data['SpeakerName'].str.strip()
data['PreparedOrQA'] = (
    data['PreparedOrQA'].str.strip().str.lower()
)

data = data[
    data['Text'].str.strip().ne('')
].sort_values(['EarningsCallID', 'SpeakerTurnID'])

excluded_speaker = data['SpeakerName'].str.contains(
    r'operator|unknown|analyst',
    case=False, regex=True
)

prepared = data[
    data['PreparedOrQA'].eq('prepared') & ~excluded_speaker
].copy()

question_cue = re.compile(
    r"\?|\b(?:question|wondering|could you|can you|"
    r"would you|help us|talk about|walk us|"
    r"give us|elaborate|clarify)\b", re.I
)

answer_indices = []

for call_id, turns in data.groupby('EarningsCallID', sort=False):
    managers = set(
        prepared.loc[
            prepared['EarningsCallID'].eq(call_id),
            'SpeakerName'
        ]
    )

    answering_question = False

    for idx, turn in turns.iterrows():
        if turn['PreparedOrQA'] != 'qa':
            continue

        speaker = turn['SpeakerName']

        if 'operator' in speaker.lower():
            answering_question = False
        elif speaker not in managers:
            answering_question = bool(
                question_cue.search(turn['Text'])
            )
        elif answering_question:
            answer_indices.append(idx)

qa_answers = data.loc[answer_indices].copy()

prepared['Class'] = 'Prepared'
qa_answers['Class'] = 'Q&A'

# Combine labeled observations
dataset = pd.concat(
    [prepared, qa_answers],
    ignore_index=True
)

# 80% training / 20% testing by earnings call
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        dataset,
        groups=dataset['EarningsCallID']
    )
)

train_df = dataset.iloc[train_idx].copy()
test_df = dataset.iloc[test_idx].copy()

assert set(train_df['EarningsCallID']).isdisjoint(
    set(test_df['EarningsCallID'])
)

assert train_df['Class'].nunique() == 2
assert test_df['Class'].nunique() == 2

# Train Naive Bayes on training data ONLY
train_prepared = train_df[
    train_df['Class'] == 'Prepared'
]
train_qa = train_df[
    train_df['Class'] == 'Q&A'
]

PreparedCounts = Counter(
    token
    for text in train_prepared['Text']
    for token in words(text)
)

QACounts = Counter(
    token
    for text in train_qa['Text']
    for token in words(text)
)

PreparedTotal = sum(PreparedCounts.values())
QATotal = sum(QACounts.values())

Vocabulary = set(PreparedCounts) | set(QACounts)

# Preserve equal class priors
Priors = {'Prepared': 0.5, 'Q&A': 0.5}

print("Training calls:",
      train_df['EarningsCallID'].nunique())
print("Testing calls:",
      test_df['EarningsCallID'].nunique())

print("Training observations:", len(train_df))
print("Testing observations:", len(test_df))

print("Training percentage:",
      round(len(train_df) / len(dataset) * 100, 2))

print("Testing percentage:",
      round(len(test_df) / len(dataset) * 100, 2))

print("\nClass distribution:")
display(pd.crosstab(
    dataset.loc[train_idx.tolist() + test_idx.tolist(), 'Class'],
    np.array(['Training'] * len(train_idx) +
             ['Testing'] * len(test_idx)),
    rownames=['Class'],
    colnames=['Split']
))

print("Vocabulary size:", len(Vocabulary))
print("Model training complete.")


Training calls: 160
Testing calls: 40
Training observations: 3706
Testing observations: 888
Training percentage: 80.67
Testing percentage: 19.33

Class distribution:


Split,Testing,Training
Class,,
Prepared,243,1126
Q&A,645,2580


Vocabulary size: 16068
Model training complete.


In [3]:
newSentence = "Yeah, thanks for the question."

def classify_sentence(sentence):
    tokens = words(sentence)
    counts = Counter(tokens)
    scores = {label: math.log(prior) for label, prior in Priors.items()}
    rows = []
    for word, frequency in counts.items():
        if word not in Vocabulary:
            continue
        # Multinomial Naive Bayes with Laplace (add-one) smoothing.
        p_prepared = (PreparedCounts[word] + 1) / (PreparedTotal + len(Vocabulary))
        p_qa = (QACounts[word] + 1) / (QATotal + len(Vocabulary))
        prepared_log = frequency * math.log(p_prepared)
        qa_log = frequency * math.log(p_qa)
        scores['Prepared'] += prepared_log
        scores['Q&A'] += qa_log
        rows.append({
            'Word': word, 'Sentence count': frequency,
            'Prepared training count': PreparedCounts[word],
            'QA training count': QACounts[word],
            'P(word | Prepared)': p_prepared, 'P(word | QA)': p_qa,
            'Prepared log contribution': prepared_log,
            'QA log contribution': qa_log,
            'Prepared minus QA': prepared_log - qa_log
        })
    maximum = max(scores.values())
    weights = {label: math.exp(score - maximum) for label, score in scores.items()}
    probabilities = {label: weight / sum(weights.values()) for label, weight in weights.items()}
    prediction = max(scores, key=scores.get)
    if not rows or math.isclose(scores['Prepared'], scores['Q&A'], abs_tol=1e-12, rel_tol=0):
        prediction = 'Unclassified (no known words or tied scores)'
    details = pd.DataFrame(rows)
    if rows:
        details = details.loc[details['Prepared minus QA'].abs().sort_values(ascending=False).index]
    return prediction, scores, probabilities, details, tokens

Prediction, Scores, Probabilities, WordDetails, SentenceTokens = classify_sentence(newSentence)
print('Sentence:', newSentence)
print('Prediction:', Prediction)
#print('Score = log(prior) + sum(count * log(P(word | class)))')
display(pd.DataFrame({
    'Class': list(Scores),
    'Log score (higher wins)': list(Scores.values()),
    'Normalized model probability': [Probabilities[label] for label in Scores]
}))
print('Recognized tokens:', sum(word in Vocabulary for word in SentenceTokens), '/', len(SentenceTokens))
print('Ignored unseen words:', sorted(set(SentenceTokens) - Vocabulary))
print('Positive Prepared minus QA favors Prepared; negative favors Q&A.')
display(WordDetails.round(6))

# Basic behavior checks: no known words must not produce a confident label.
assert classify_sentence('12345 !!!')[0].startswith('Unclassified')
assert math.isclose(sum(Probabilities.values()), 1.0)


from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# Predict on unseen test observations
y_true = test_df['Class'].tolist()

y_pred = [
    classify_sentence(text)[0]
    for text in test_df['Text']
]

labels = ['Prepared', 'Q&A']

print("TEST SET EVALUATION")
print("===================")

accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0
))

print("\nConfusion Matrix:")
cm = confusion_matrix(
    y_true, y_pred, labels=labels
)

display(pd.DataFrame(
    cm,
    index=['Actual Prepared', 'Actual Q&A'],
    columns=['Predicted Prepared', 'Predicted Q&A']
))

print("\nUnclassified predictions:",
      sum(p not in labels for p in y_pred))


Sentence: Yeah, thanks for the question.
Prediction: Q&A


,Class,Log score (higher wins),Normalized model probability
0,Prepared,-32.655460,0.012637
1,Q&A,-28.297026,0.987363


Recognized tokens: 5 / 5
Ignored unseen words: []
Positive Prepared minus QA favors Prepared; negative favors Q&A.


,Word,Sentence count,Prepared training count,QA training count,P(word | Prepared),P(word | QA),Prepared log contribution,QA log contribution,Prepared minus QA
0,yeah,1,64,1035,0.000132,0.002323,-8.932869,-6.064844,-2.868024
4,question,1,198,665,0.000404,0.001493,-7.813951,-6.506677,-1.307274
1,thanks,1,284,365,0.000579,0.000821,-7.454767,-7.105333,-0.349433
2,for,1,4815,3515,0.009779,0.007884,-4.627557,-4.842887,0.215330
3,the,1,21462,20410,0.043579,0.045769,-3.133170,-3.084137,-0.049033


TEST SET EVALUATION
Accuracy: 0.8626

Classification Report:
              precision    recall  f1-score   support

    Prepared       0.83      0.63      0.72       243
         Q&A       0.87      0.95      0.91       645

   micro avg       0.86      0.86      0.86       888
   macro avg       0.85      0.79      0.81       888
weighted avg       0.86      0.86      0.86       888


Confusion Matrix:


,Predicted Prepared,Predicted Q&A
Actual Prepared,154,89
Actual Q&A,32,612



Unclassified predictions: 1
